# Tokenization Tutorial
## Made Using Karpathy's video, not as detailed.

(The video mas made in early 2024, so some points may not stand with modern language models.)

"Language Models are Unsupervised Multi-task learners" - GPT-2 Paper, that introduces BPE algorithm for tokenization in LLMs.

## Some Problems in LLMs:

- Why cannot LLMs spell words?
- Why cannot LLMs do simple string processing (like reverse a string?)
- LLMs are worse at non-English language\.
- GPT-2 had trouble in coding in python\.
- LLMs end abruptly upon seeing '< |endoftext| >'
- LLMs give 'trailing whitespace' warning\.
- The word 'SolidGoldMagicKarp'\.
- YAML over JSON?
- LLMs are not actually end-to-end language modelling\.

Most Importantly:
**LLMs are bad at simple arithmetic.**

In Python, strings (text) are *immutable sequences of 'unicode' code points*.

**Unicode** is a definition of ~150K characters and 161 scripts.
For example,
'a' is defined as U+0061 (it uses hexadecimal, value is 0061 in hexadecimal, which is 97 in decimal.)

So the simplest naive tokenizer can be:

In [8]:
sentence = "Hello World!"
naive_tokens = [ord(x) for x in sentence]
naive_tokens

[72, 101, 108, 108, 111, 32, 87, 111, 114, 108, 100, 33]

Why is this not used?

- *Unicode has too large of vocabulary*.
- *Unicode keeps changing*.

So we move to encodings:

Unicode defines 3 encodings:
- *UTF-8* -> most commonly used.
- *UTF-16*
- *UTF-32*

UTF-8: for each code point, it uses between 1 and 4 bytes to represent as byte representation.
UTF-32: for each code point, it uses exactly 32 bits (4 bytes).

In [9]:
simple_encoding = sentence.encode("utf-8")

print(simple_encoding, type(simple_encoding))

# byte class cannot really be used for any further processing, so we convert to a list of numbers.

print(list(simple_encoding))

b'Hello World!' <class 'bytes'>
[72, 101, 108, 108, 111, 32, 87, 111, 114, 108, 100, 33]


In [10]:
new_sentence = "I am <name>!"

list(new_sentence.encode("utf-8"))

[73, 32, 97, 109, 32, 60, 110, 97, 109, 101, 62, 33]

Above, each 'a' char is represented by the number 97. Basically, 'a'  ->  U+0061  ->  UTF-8  ->  01100001  ->  we read as 97.
Basically, using utf-8, memory stores 01100001 in the byte to store 'a'.

But, even this encoding does not really work, because naive utf-8 encoding only means 256 unique tokens (1 byte = 8 bits, 2^8 = 256, so only all numbers from 0 to 255.) This means a small embedding table for the LLM, but a large sequence length because of the small vocab size, and then we cannot really attend properly to all tokens, also context is lost, resulting in bad performance and output.

So the solution is **(BPE - (Byte Pair Encoding))**.

Suppose, initially

Text is "aaabdaaabac", and vocab is {a, b, c, d}

We find the most commonly occurring pair of tokens/chars, which in this case is "aa".

Now, we replace "aa" with a char not present in the vocabulary currently, say 'Z'.

so,
Text:
ZabdZabac,
Z=aa,
vocab = {a, b, c, d, Z}.

Again, most common pair = "ab"
So, 

ZYdZYac,
Z=aa,
Y=ab,
vocab = {a, b, c, d, Z, Y}

most common pair = "ZY"
So,

XdXac,
Z=aa,
Y=ab,
Z=XY,
vocab = {a, b, c, d, Z, Y, X}

We have now increase vocab size from 4 to 7 and decreased longest sequence length from 11 to 5.

In [11]:
# A very naive implementation of this:
import string
import random

all_characters = list(string.ascii_lowercase)

sample_text = "aaabdaaabac"
current_vocab = set(sample_text)

def get_most_frequent(string_, current_vocab):
    
    freq_map = {}
    
    for pair in zip(string_, string_[1:]):
        freq_map[pair] = freq_map.get(pair, 0) + 1
        
    top_pair = max(freq_map, key=freq_map.get) # most commonly occurring pair.
    
    replacement_set = list(set(all_characters) - current_vocab)
    replacement_char = random.choice(replacement_set)
    
    return (top_pair, freq_map[top_pair], replacement_char)

# main loop
while True:
    top_pair, freq, replacement_char = get_most_frequent(sample_text, current_vocab)
    
    if freq > 1:
        current_vocab.add(replacement_char)
        top_pair = "".join(top_pair)
        print(f"Top Pair: {top_pair}")
        
        sample_text = sample_text.replace(top_pair, replacement_char)
        print(f"New Text: {sample_text}")
        print(f"New Vocab: {current_vocab}")        
            
    else:
        break

Top Pair: aa
New Text: rabdrabac
New Vocab: {'c', 'b', 'r', 'a', 'd'}
Top Pair: ra
New Text: vbdvbac
New Vocab: {'c', 'b', 'r', 'a', 'd', 'v'}
Top Pair: vb
New Text: zdzac
New Vocab: {'c', 'b', 'z', 'r', 'a', 'd', 'v'}


What this does for a large amount of text with millions of words:
initially, "cat" is c, a, t.
Then "ca" or "at" convert to a single token.
then cX or Xt convert again to a single token.
All said and done, common English words like cat, there, their, home are always represented as a single token in the model.

This is also the reason language models perform considerably worse on other languages, because the training corpus has a high quantity of English content compared to any other language, so commonly used English words are represented by a single token, but, say, commonly used Hindi words remain as separate tokens. What this does is for a Hindi input text, the sequence length is very large compared to English, and we cannot attend properly to each token, and again context is lost, and performance is bad.

Hence, it is better to include good percentages of all languages in the training corpus.

## Let's do this properly on a sizable text.

In [12]:
text = """Abel Tesfaye[a] (born Abel Makkonen Tesfaye;[b] February 16, 1990), known professionally as the Weeknd, is a Canadian singer-songwriter, record producer, and actor.[3][4] Regarded as an influential figure in popular music, he is known for his light-lyric tenor vocal range and falsetto, alternative R&B sound, and dark aesthetics, as well as the cinematic visuals and storytelling of his music videos. His accolades include 4 Grammy Awards, 20 Billboard Music Awards, 22 Juno Awards, 6 American Music Awards, 3 MTV Video Music Awards, and a Latin Grammy Award.

Tesfaye began releasing music anonymously in 2009. After co-founding the record label XO, he released three mixtapes—House of Balloons, Thursday, and Echoes of Silence—in 2011. He signed with Republic Records to compile the mixtapes into the compilation album Trilogy (2012), before releasing his debut studio album, Kiss Land (2013). Following collaborations and film soundtrack contributions from 2013 and 2014, Tesfaye blended alternative R&B with pop on his second and third studio albums, Beauty Behind the Madness (2015) and Starboy (2016); both debuted atop the US Billboard 200 and featured the Billboard Hot 100 number-one singles "Can't Feel My Face", "The Hills", "Starboy", and "Die for You". His debut extended play (EP), My Dear Melancholy (2018), featured the US top-ten single "Call Out My Name".

Tesfaye began an album trilogy based on three time points, starting with the dream pop– and new wave–inspired album After Hours (2020), which spawned the chart-topping singles "Heartless" and "Save Your Tears", as well as "Blinding Lights"—the best-performing song in the Billboard Hot 100's history and the most-streamed song on Spotify. The trilogy's latter two installments, Dawn FM (2022) and Hurry Up Tomorrow (2025), featured the US top-ten singles, "Take My Breath" and "Timeless".

Tesfaye is one of the best-selling and most-streamed artists of all time. He has amassed eight diamond-certified singles from the Recording Industry Association of America (RIAA).[5] The first artist to ever surpass 100 million monthly listeners on Spotify and the highest-paid musician in 2025, he ranked among the world's 100 most influential people by Time in 2020.[6] His After Hours til Dawn Tour (2022–2026) set the record for the highest-grossing tour by a male soloist in history.[7] Tesfaye also co-created and starred in the HBO drama series The Idol (2023), and the film Hurry Up Tomorrow (2025). His other ventures include hosting the Apple Music 1 radio show Memento Mori (2018–2022), being appointed goodwill ambassador for the World Food Programme in 2021, and advocacy for racial equality and food security.

Life and career
1990–2008: Early life
Abel Tesfaye (born Abel Makkonen Tesfaye) was born on February 16, 1990, in a city in Ontario, variously reported to be Toronto,[c] or Scarborough.[d][e] The only child of Ethiopian immigrants Makkonen Tesfaye and Samrawit Hailu,[18] who separated shortly after his birth,[3] he was raised in the suburb of Scarborough by his mother and grandmother.[19][20] Tesfaye's patronymic is spelled "Makkonen" instead of the traditional Ethiopian name "Makonnen". The similarity with the Finnish surname Makkonen is pure coincidence. The spelling of Tesfaye's patronymic might be the result of a typographic error or a new form of the traditional name.[21] In 2024 he legally removed his middle name.[22] Tesfaye grew up speaking Amharic,[23] and is also fluent in French, as he attended a French immersion school.[24] He was further educated at West Hill Collegiate Institute and Birchmount Park Collegiate Institute.[25]

At seventeen, Tesfaye dropped out of school and relocated to an apartment in the neighbourhood of Parkdale with two friends, one of whom was La Mar Taylor—his best friend and now creative director.[26] Living a hedonistic lifestyle with his friends,[19][27] Tesfaye adopted his stage name because he left home on a weekend.[28] He removed the last 'e' in 'weekend' to avoid trademark issues with the Canadian pop rock band the Weekend.[19] He has also experienced homelessness and was incarcerated on several occasions during this time, which encouraged him to "smarten up, to focus".[29][30] During this time, Tesfaye frequently engaged in drug use, including substances such as ketamine, cocaine, MDMA, magic mushrooms, and cough syrup,[31] stating that drugs were a "crutch" for him when he wrote music.[32] Before releasing music under his current stage name, he went under the alias "Kin Kane", as part of a hip-hop duo called "Bulleez n Nerdz",[3] and was part of a production team called 'the Noise'.[33][34]

2009–2014: Trilogy and Kiss Land
The Weeknd wearing a button-up, sleeveless shirt covering a normal black t-shirt. He is performing on a stage, singing into a microphone, with palm trees in the background.
Tesfaye performing at the Coachella Valley Music and Arts Festival in 2012
In August 2009, Tesfaye began anonymously releasing music on YouTube.[35][1] The following year, he met the producer Jeremy Rose at a party. Rose asked Tesfaye if he wanted to work together as a dark R&B project after hearing him freestyle over an instrumental. After creating multiple songs and parting ways due to creative differences, Tesfaye was allowed to use the songs they made together under the condition that Rose received production credits.[34] In December 2010, Tesfaye uploaded "What You Need", "Loft Music" and "The Morning" to YouTube under the username "xoxxxoooxo".[36][37] His identity remained undisclosed initially. These songs gained attention online and were later acknowledged in a blog post by the rapper Drake.[34][38] The songs subsequently received coverage from various media outlets, including Pitchfork and The New York Times.[39]

In 2011, Tesfaye met music executives Wassim "Sal" Slaiby and Amir "Cash" Esmailian, with whom, along with Taylor, he founded the XO record label; earlier in 2009, Tesfaye and Taylor, along with friend and later photographer and collaborator Hyghly Alleyne formed a multimedia collective, She's So Lovely via Tumblr.[40] On March 21, Tesfaye released his debut mixtape, House of Balloons,[41][42][43] which featured production from Illangelo and Doc McKinney. The mixtape also included tracks produced by Rose, although he did not receive production credits.[34] House of Balloons was named as one of the ten shortlisted nominees for the 2011 Polaris Music Prize.[44][45] Tesfaye started working with Drake in May, eventually earning a spot at the latter's OVO Fest on July 31.[46] That month, Tesfaye held his first live performance at the Mod Club Theatre in Toronto, which received media attention for Drake being in attendance.[47][45] He also participated in concerts hosted by the Black Student Association at the University of Toronto.[1] In summer 2011, he also received media attention in the United States when his song "High For This" was featured in an ad campaign for the final season of HBO's "Entourage."[45] On August 18, Tesfaye released his second mixtape, Thursday, which garnered usually positive reviews.[48] Tesfaye contributed to four songs on Drake's second studio album, Take Care, released on November 15, as a songwriter, producer and a featured artist on the album's seventh single, "Crew Love".[49] He released his third mixtape, Echoes of Silence, on December 21. It was a long-listed nominee for the 2012 Polaris Music Prize.[50][51][45]

In April 2012, Tesfaye began performing at more shows, such as the Coachella Festival,[52] and two sold-out shows at the Bowery Ballroom in New York City.[52][53] He also performed at various European festivals, such as Primavera Sound in Spain and Portugal[54] and the Wireless Festival in the United Kingdom.[55][56][57] In September, Tesfaye signed with Republic Records; XO was assumed as a subsidiary label.[58] That same month, he embarked on his first concert tour, the Fall Tour, which included his own headlining shows and some opening shows for the English band Florence and the Machine. The tour was performed in North America in September to November.[59] On November 13, Tesfaye released Trilogy, a compilation album comprising re-mixed and remastered versions of his 2011 mixtapes, and three additional tracks.[60] The album debuted at number four on the US Billboard 200 with first-week sales of 86,000 copies,[61] and has received platinum certifications from the Recording Industry Association of America (RIAA) and double-platinum from Music Canada.[62][63] It also earned Tesfaye a nomination for the Sound of 2013 poll award by BBC.[64]

The picture depicts Tesfaye performing, with the lights giving the image an orange environment. In the background, there is a drum set.
Tesfaye performing at Massey Hall in October 2013
On May 17, 2013, Tesfaye released the title track to his debut studio album, Kiss Land[65] and announced the album's release date of September 10.[66] Upon its release, the album debuted at number two on the Billboard 200 with 96,000 copies[67] and received generally positive reviews from music critics.[68] Tesfaye further promoted the album with a fall tour that occurred in North America and England in September to November.[69] In October 2013, Tesfaye was announced as the opening act for Drake's European leg of Would You Like a Tour?, lasting between February and March 2014.[70] Between November 6 and 13, he served as an opening act for Justin Timberlake during The 20/20 Experience World Tour.[71] He also contributed two songs to the soundtrack for the 2013 film The Hunger Games: Catching Fire, "Devil May Cry" and the soundtrack's second single, "Elastic Heart" with Sia and Diplo.[72]

In February 2014, Tesfaye released a remix of "Drunk in Love" from Beyoncé's eponymous studio album,[73] and Ty Dolla Sign's "Or Nah".[74] He announced the King of the Fall Tour in June, a 4-city tour of North America between September and October and was supported by Schoolboy Q and Jhené Aiko.[75] In promotion of the tour, he released the songs "King of the Fall" and "Often" in July of that year.[76] On August 25, Tesfaye collaborated with Ariana Grande on the song "Love Me Harder" from Grande's second studio album My Everything. It was later released on September 30 as the fourth single from the album, and peaked at number seven on the Billboard Hot 100.[77] On December 23, Tesfaye released the song "Earned It" from the soundtrack for Fifty Shades of Grey (2015). The single, which peaked at number three on the Billboard Hot 100, earned Tesfaye his first and only Academy Award nomination for Best Original Song.[78] The song won Best R&B Performance and was nominated for Best R&B Song and Best Song Written for Visual Media at the 58th Annual Grammy Awards.[79]

2015–2016: Beauty Behind the Madness
The Weeknd, sporting a Basquiat-like hairstyle, holds a microphone while his other hand points up as he performs.
Tesfaye performing at Bumbershoot, 2015
On May 27, 2015, Tesfaye released the second single from Beauty Behind the Madness, "The Hills".[80] The single debuted at number twenty on the Billboard Hot 100, and peaked at number one, becoming Tesfaye's second number-one single, following "Can't Feel My Face", which had reached the number one position before it, and by four years, was certified diamond by the RIAA, marking Tesfaye's first diamond-certified record.[81][82] In June 2015, after winning the Centric Award at the BET Awards, Tesfaye performed "Earned It" with Alicia Keys.[83] On June 8, he released the song "Can't Feel My Face" as the album's third single. The track was previously leaked in May, but was released as a single following a performance by Tesfaye at the Apple Worldwide Developers Conference.[84] The single debuted at number twenty-four on the Billboard Hot 100, and peaked at number one, making it Tesfaye's third top 10 hit and his first number-one song in the United States.[85][86] The song was nominated for Record of the Year and Best Pop Solo Performance at the 58th Annual Grammy Awards.[87]

Tesfaye occupied all three slots on Billboard's Hot R&B/Hip-Hop Songs chart simultaneously with the aforementioned singles, becoming the first artist in history to accomplish this.[88] He was also unveiled as one of the musical faces of the streaming service Apple Music, alongside Drake.[89] During the 2015 MTV Video Music Awards, Apple debuted a two-part promotional commercial featuring Tesfaye, which had a guest appearance from John Travolta.[90] In July, Tesfaye headlined the inaugural FVDED in the Park festival in Surrey, British Columbia.[91] On June 29, Tesfaye was featured on Meek Mill's second studio album Dreams Worth More Than Money (2015), on the track "Pullin' Up".[92]

Beauty Behind the Madness, Tesfaye's second studio album, was released on August 28, 2015, and debuted atop the Billboard 200, earning 412,000 album-equivalent units in its first week.[93][94] It reached the top 10 in over ten countries and reached number one in Canada, Australia, Norway, and the United Kingdom.[95][96] The album was promoted by Tesfaye headlining various summer music festivals, including Lollapalooza, the Hard Summer Music Festival, and the Bumbershoot Festival.[97] He announced The Madness Fall Tour, his first large-scale tour across the United States, which began in November, and concluded in December.[98][99] The album was certified double platinum in the U.S., and sold 1.5 million copies worldwide.[100] It was the most-streamed album in 2015, with over 60 million streams,[101] and was ranked on multiple lists of albums of the year.[102] The three singles that preceded the album were certified platinum in the United States.[103] The album won Best Urban Contemporary Album and was nominated for Album of the Year at the 58th Annual Grammy Awards.[104]

On September 4, 2015, Tesfaye was featured on Travis Scott's debut album Rodeo, on the track "Pray 4 Love".[105] On October 10, Tesfaye appeared on Saturday Night Live alongside actress Amy Schumer, performing as the show's musical guest.[106][107] This was his first performance on the show as a solo artist, after appearing with Ariana Grande to perform "Love Me Harder".[107] In November, he began his debut arena tour, The Madness Fall Tour, concluding in December.[98] On December 18, Tesfaye was featured on Belly's single "Might Not" from his eighth mixtape Up For Days.[108] On February 14, 2016, Tesfaye was featured on Kanye West's seventh studio album The Life of Pablo on the track "FML".[109][110] It marked their second collaboration, with West previously writing and producing on Tesfaye's track "Tell Your Friends" from Beauty Behind the Madness.[111] On March 1, Tesfaye was featured on Future's single "Low Life" from his fourth studio album Evol.[112] On April 23, he was featured on Beyoncé's sixth studio album Lemonade on the track "6 Inch".[113] On August 26, Tesfaye was featured on Cashmere Cat's single "Wild Love" with Francis and the Lights, which served as the lead single from Cashmere Cat's debut studio album 9 (2017).[114]

2016–2019: Starboy and My Dear Melancholy,
The Weeknd looks slightly up in a pink-lit environment during a concert
Tesfaye performing at Lollapalooza Chile during the Starboy: Legend of the Fall Tour in 2017
In September 2016, Tesfaye announced that his third studio album, Starboy, would be released on November 25, and included collaborations with now-disbanded French electronic music duo Daft Punk.[115][116] He released the album's title track, which featured the duo on September 21.[117] The song debuted at number 40 on the Billboard Hot 100, and peaked at number one, making it Tesfaye's third number-one single.[118] As of March 2023, the song is certified Diamond by the RIAA.[119] Their second collaboration, "I Feel It Coming" was released on November 24. The single peaked at number four on the Billboard Hot 100.[120] On October 1, Tesfaye made a second appearance on Saturday Night Live as the musical guest alongside actress Margot Robbie. During the show, he performed "Starboy" and "False Alarm".[121] On November 23, he released the short film M A N I A. Directed by Grant Singer, it featured excerpts from the album, including snippets from "All I Know" featuring Future, "Sidewalks" featuring Kendrick Lamar, "Secrets", and "Die for You".[122] Upon release, the album debuted at number one on the U.S. Billboard 200 with 348,000 units, making it Tesfaye's second consecutive number-one album.[123] As of January 2019, the album is certified triple platinum by the RIAA.[119] The album won Best Urban Contemporary Album at the 60th Annual Grammy Awards, making it Tesfaye's second win in the category.[124]

On February 17, 2017, Tesfaye began his fifth concert tour, called Starboy: Legend of the Fall Tour.[125] The tour was in support of his third studio album Starboy, and concluded on December 14.[126] He visited the continents Europe,[125] North America,[125] and Oceania.[126] On February 15, Tesfaye was featured on Nav's commercial debut single "Some Way", which also served as the lead single from his self-titled mixtape.[127] On February 24, he appeared on Future's sixth studio album Hndrxx, on the song "Comin Out Strong".[128] On April 19, Tesfaye appeared on the title track and second single from Lana Del Rey's fifth studio album.[129][130] On July 30, he was featured on French Montana's track "A Lie", the third single from his second studio album Jungle Rules.[131] He then appeared on the Virgil Abloh-directed music video for Lil Uzi Vert's "XO Tour Llif3" alongside Nav. He was later featured on Lil Uzi Vert's debut album Luv Is Rage 2 on the track "UnFazed"[132] and on Gucci Mane's eleventh studio album Mr. Davis on the track "Curve".[133]

On February 2, 2018, Tesfaye contributed to the soundtrack for Black Panther on the song "Pray for Me" with Kendrick Lamar. The track served as the third single from the soundtrack, and peaked at number seven on the Billboard Hot 100.[134][135] On March 30, Tesfaye released his debut extended play My Dear Melancholy,[136][137] after news of the project were teased and leaked.[138][139] The EP debuted at number one on the Billboard 200 with 169,000 units, making it Tesfaye's third consecutive number-one album and the shortest album, by track count, to top the chart in eight years.[140] On April 6, Tesfaye released the EP's lead single "Call Out My Name", which peaked at number four on the Billboard Hot 100.[141][142] On April 13, he headlined the Coachella Valley Music and Arts Festival for the first time.[143][144] He appeared in multiple festivals throughout 2018 to support the EP, most notably the Mawazine Festival in Morocco,[145] Lollapalooza in both Chicago and Berlin,[146] and a post-race concert at the Abu Dhabi Grand Prix.[147]

On June 6, 2018, Tesfaye announced his new Apple Music 1 radio show Memento Mori. The first episode was released two days later.[148] He would later feature on two tracks from Travis Scott's third studio album, Astroworld, which were "Skeletons" and "Wake Up" on August 3.[149] On November 21, he released his first greatest hits album The Weeknd in Japan.[150] In support of the album and his EP My Dear Melancholy, he began his sixth concert tour, the Weeknd Asia Tour, a six-show tour of Asia during November and December.[151] On January 11, 2019, Tesfaye was featured on Gesaffelstein's song "Lost in the Fire", the second single from his second studio album Hyperion.[152] On April 18, he released "Power Is Power" with SZA and Travis Scott, the lead single from the Game of Thrones-inspired soundtrack.[153][154] On August 30, during the Telluride Film Festival, he made his acting debut in the film Uncut Gems as himself.[155]

2019–2021: After Hours and Super Bowl LV halftime show
On November 24, 2019, Tesfaye teased his single "Blinding Lights" through a Mercedes-Benz commercial.[156] On November 27, he released "Heartless" as the lead single from his fourth studio album. The song debuted at number thirty-two on the Billboard Hot 100 and peaked at number one, making it Tesfaye's fourth number-one single.[157][158] "Blinding Lights" was released two days after the release of "Heartless" on November 29. The single debuted at number eleven on the Billboard Hot 100 and peaked at number one, making it Tesfaye's fifth number-one single.[159] "Blinding Lights" would then go on to become the first song in the chart's history to hold a spot in the top ten for an entire year.[160] It also became the longest charting song by a solo artist on the Hot 100 at 90 weeks, ending the week of September 11, 2021.[161][162] On November 23, 2021, "Blinding Lights" was ranked as the best-performing song in the Billboard Hot 100's history, surpassing "The Twist" by Chubby Checker.[163] On January 1, 2023, it became the most streamed song on Spotify with 3.3 billion streams.[164]

On February 19, 2020, Tesfaye revealed that his fourth studio album would be titled After Hours, and would be released on March 20. He also released the album's title track as a promotional single.[165] On March 7, he made his third appearance as a musical guest on Saturday Night Live, alongside actor Daniel Craig. On the show, he starred in the skit "On The Couch" with actors Kenan Thompson and Chris Redd,[166] performed "Blinding Lights", and debuted the track "Scared to Live".[167] Tesfaye released the album's third single "In Your Eyes" on March 24. The track peaked at number sixteen on the Billboard Hot 100.[168] Upon release,[169] After Hours debuted atop the Billboard 200, earning 444,000 units, marking Tesfaye's fourth consecutive number-one album.[170] It became the most streamed R&B album of all time, surpassing Tesfaye's own Starboy.[171][172] In the album's first charting week, Tesfaye also became the first artist to lead the Billboard 200, Billboard Hot 100, Billboard Artist 100, Hot 100 Songwriters and Hot 100 Producers charts simultaneously, and repeated his lead the following week.[173][174] The deluxe version of After Hours was released on March 29, 2020, and contained the tracks "Nothing Compares", "Missed You" and "Final Lullaby".[175] On May 4, Tesfaye made a guest appearance on the American Dad! episode "A Starboy Is Born",[176] also co-written by Tesfaye and featuring a song titled "The Weeknd's Dark Secret".[177] On July 27, he voiced three characters during the 200th episode of Robot Chicken.[178]

On August 7, 2020, Tesfaye was featured on the late Juice Wrld's single "Smile" from his first posthumous album Legends Never Die.[179] On August 28, he was featured on Calvin Harris' single "Over Now".[180] On October 30, Tesfaye appeared on Ariana Grande's song "Off the Table" from her sixth studio album Positions.[181][182] On the same day, he appeared on Oneohtrix Point Never's track "No Nightmares" from his ninth studio album Magic Oneohtrix Point Never, which he also executive produced with OPN.[183] On November 5, he appeared on the remix of Maluma's "Hawái", was nominated for Best Urban Fusion/Performance at the 22nd Annual Latin Grammy Awards.[184][185] On December 10, he performed at iHeartRadio's Jingle Ball.[186] On February 5, 2021, Tesfaye released his second greatest hits album The Highlights.[187] The album debuted at number two on the US Billboard 200, making it Tesfaye's highest charting compilation album and the biggest first week debut for a greatest hits album since Fully Loaded: God's Country (2019).[188]

An aerial view of the Raymond James Stadium at daylight in 2021
Tesfaye headlined the Super Bowl LV halftime show at the Raymond James Stadium in 2021
Tesfaye headlined the Super Bowl LV halftime show on February 7, 2021, becoming the first Canadian solo artist to headline the show.[189][190][191][192] He reportedly spent US$7 million of his own money on the Super Bowl performance.[40] Reviews of the performance were generally positive.[193][194][195][196][197][198] The show resulted in a surge in streaming and downloads for Tesfaye's After Hours album as well as for the seven other songs he performed.[199][200] The halftime show earned three nominations at the 73rd Primetime Emmy Awards: Outstanding Variety Special (Live), Outstanding Lighting Design/Lighting Direction for a Variety Special, and Outstanding Technical Direction, Camerawork, Video Control for a Special.[201]

Over 2021, Tesfaye reissued his three mixtapes in its authentic form with the original mixes and samples to celebrate the tenth anniversary of their release, with House of Balloons coming first in March.[202] Thursday's reissue followed in August,[203][204] and in December, Echoes of Silence was reissued.[205][206] On April 23, Tesfaye released a remix of "Save Your Tears" with Ariana Grande, marking their third collaboration.[207] The remix launched the song to the top of the Billboard Hot 100 on the chart dated May 8, 2021, becoming both artists' sixth number one hit.[208] He later began to tease new music in the same month. When asked about a new album during an interview with Variety, he explained that "if the last record is the After Hours of the night, then The Dawn is coming".[209] On May 11, Tesfaye performed "Save Your Tears" at the Brit Awards. He also accepted his first Brit Award for International Male Solo Artist, which was presented to him by former first lady of the United States Michelle Obama.[210][211]

On May 24, Tesfaye performed "Save Your Tears" at the Billboard Music Awards. He was nominated for a record sixteen awards, and won ten, including Top Artist and Top Hot 100 Song. When accepting his awards, Tesfaye continued to tease new music by saying "the After Hours are done, and The Dawn is coming".[212] On May 28, he performed the remix of "Save Your Tears" at the iHeartRadio Music Awards with Ariana Grande.[187] On June 25, Tesfaye appeared on Doja Cat's single "You Right" from her third studio album Planet Her.[213] On July 22, he appeared on Belly's single "Better Believe" with Young Thug from his third studio album See You Next Wednesday.[214]

2021–2023: Dawn FM
An extreme wide shot of the Weeknd performing with a durag in white-colored fashion as he sits down and performs on the staircase of the concert's setup.
Tesfaye in July 2023 during his After Hours til Dawn Tour in Paris.
On August 2, 2021, Tesfaye released a snippet of new music on social media.[215] He appeared on the cover of the September 2021 issue of GQ, marking the magazine's first global publication.[216][217] Then, in a collaboration with NBC Sports and the 2020 Summer Olympics, Tesfaye announced the single "Take My Breath", which was released on August 6.[218][219] Later that month, he appeared on Kanye West's tenth studio album Donda on the track "Hurricane", which won Best Melodic Rap Performance at the 64th Annual Grammy Awards.[220] On October 4, during an episode of Memento Mori, Tesfaye revealed that his fifth studio album was complete and that he was waiting on a "couple characters that are key to the narrative."[221] On October 18, Tesfaye announced that his upcoming tour, originally titled the After Hours Tour, would be held entirely in stadiums due to arena constraints and was scheduled to commence in July 2022.[222] The tour was renamed as the After Hours til Dawn Tour, and would incorporate elements from his fourth and fifth studio albums.[223][224]

On October 22, 2021, Tesfaye appeared on Swedish House Mafia's single "Moth to a Flame" from their debut studio album Paradise Again.[225] On November 5, he appeared on Post Malone's single "One Right Now" from his fourth studio album Twelve Carat Toothache.[226][227] On November 11, he was featured on Rosalía's single "La Fama" from her third studio album Motomami.[228][229] On December 16, Tesfaye was featured on FKA Twigs' single "Tears in the Club" from her debut mixtape Caprisongs.[230][231] The next day, on December 17, he was featured on Aaliyah's single "Poison" from her posthumous album Unstoppable.[232][233] Tesfaye released his fifth studio album Dawn FM on January 7, 2022.[234] Upon release, the album debuted at number two on the Billboard 200 with 148,000 units, marking Tesfaye's eighth top ten entry and his second non-consecutive album to debut at number two.[235][236] He also broke the record for the most simultaneous entries for a male soloist on the Billboard Global 200, with twenty-four songs on the chart.[237][238] In addition to "Take My Breath", Dawn FM was supported by the singles "Sacrifice" and "Out of Time".[239][240] On February 26, Tesfaye premiered The Dawn FM Experience, a television music special in partnership with Amazon Prime Video.[241]

A night photograph of the entrance to the Weeknd's haunted house during Universal's Halloween Horror Nights, sported in red.
The Weeknd's haunted house as part of Universal's Halloween Horror Nights.
On March 20, 2022, Tesfaye played in an episode of the cartoon The Simpsons.[242] On April 18, he headlined the Coachella Valley Music and Arts Festival for the second time, performing alongside Swedish House Mafia.[243][244] On July 8, his record of most number-one songs on the Billboard Hot R&B Songs chart was surpassed by fellow singer and would-be rival Chris Brown; Weeknd had 71 while Brown broke his record by 8 more.[245] On July 26, Tesfaye announced that he would host a haunted house at Universal Studios Florida and Hollywood, as part of Universal's Halloween Horror Nights hosted every Halloween.[246] Tesfaye appeared on the song "Creepin'" from Metro Boomin's album, Heroes & Villains, on December 2.[247] His song "Nothing Is Lost (You Give Me Strength)", made for the film Avatar: The Way of Water's official soundtrack, was released on December 16.[248]

On February 24, 2023, following months-long renewed interest in and virality of Tesfaye's 2016 song "Die for You", which began charting in 2022 and reached a new peak of 6 on the Billboard Hot 100 6 years after its release, a remix of the song featuring Ariana Grande was released. The remix marked their fourth collaboration.[249][250] In the Billboard Hot 100 issue dated March 11, the remix reached the top of the chart, becoming both artists' seventh number one hit.[251] On February 27, in the wake of the remix's success, Tesfaye became the first artist to surpass 100 million monthly listeners on Spotify.[252] On March 3, Tesfaye released his first live album, titled Live at SoFi Stadium.[253] It featured recordings from his HBO concert film of the same name, showcasing the last concert of the North American leg of his After Hours til Dawn Tour at SoFi Stadium.[254] He subsequently featured on four songs—"Artificial Intelligence", "Defame Moi", "More Coke!!", and "Emotionless"—from Mike Dean's album 4:23, released on April 29.[255] On May 8, Tesfaye stated that he was intending to retire the moniker of "the Weeknd" in favor of performing under his birth name, or adopting a new pseudonym altogether. He explained that his upcoming album would most likely serve as his "final hurrah" under the name.[256]

2023–present: The Idol and Hurry Up Tomorrow
The Weeknd, in a tuxedo, slightly looks away from the camera.
Tesfaye at the 2023 Cannes Film Festival
Tesfaye co-created the HBO drama series The Idol with Sam Levinson, and stars in the show alongside Lily-Rose Depp.[257] The series premiered at the 2023 Cannes Film Festival out of competition,[258] where it received significant controversy for its graphic depiction of onscreen nudity and sexual content. The series debuted to widespread negative reviews from critics. The Hollywood Reporter stated that the series confirms the allegations that "Instead of subtly skewering the misogynistic and predatory nature of the business, The Idol became a forbidden love story — the stuff of a toxic man's fantasy", and called it "regressive rather than transgressive".[259] He received negative reviews for his acting, with critic Robert Daniels of The Playlist writing, "Tesfaye is also a terrible actor. He lacks the comfortability, the gravitas, charisma, and charm to increase the viability of Jocelyn being attracted to him. In most scenes, Tesfaye either hides under the cover of dim lighting, obtrusive coverage, or re-recorded dialogue dubbed into several scenes."[260] On August 28, 2023, HBO announced it had cancelled The Idol after one season.[261]

On June 8, 2023, Tesfaye announced a series of EPs featuring music from The Idol.[262] Originally intended as a soundtrack album, each EP was released following the premiere of each episode of the show, which featured collaborations with Future, Playboi Carti, Madonna, Lil Baby, Lily-Rose Depp, and Jennie from the South Korean girl group Blackpink.[263] These EPs yielded the hit singles "Popular" and "One of the Girls". Later in June he received an invitation from the Academy to become a member.[264] On July 21, Tesfaye appeared on Travis Scott's lead single "K-pop" from his fourth studio album Utopia,[265] and later appeared on another track on Scott's album, "Circus Maximus".[266] Tesfaye appeared on Diddy's single "Another One of Me" on September 15, which he called his "final feature",[267][268] despite releasing multiple collaborations with Metro Boomin and Future in 2024.[269]

On December 2, 2023, Fortnite announced that Tesfaye would be featured as the headlining artist for its Fortnite Festival gamemode and outfits of him would become available to play with on December 9. His outfits have three different variants of himself; the red suit seen throughout the promotional material for After Hours, the outfit from his 2022 Coachella performance, and two of the costumes that he was seen wearing during the After Hours til Dawn Tour.[270] Tesfaye first teased news of a follow-up album to Dawn FM in 2022, telling his fans: "[I] wonder... did you know you're experiencing a new trilogy?" via Twitter.[271] On January 8, 2024, he further teased an upcoming album, posting pictures of his last two albums and a question mark on his social media.[272] On March 22, he appeared on the track "Young Metro" from Future and Metro Boomin's collaborative album, We Don't Trust You, as well as three tracks on Future and Metro Boomin's We Still Don't Trust You album two weeks later.[269][266]

On July 17, 2024, Tesfaye announced a one-night show in São Paulo, Brazil, set to take place on September 7, 2024.[273] He announced the title of his upcoming sixth album, Hurry Up Tomorrow, three days prior to the concert,[274][275] and on September 13, released "Dancing in the Flames", which was then intended as the debut single from the album.[276] Three days later, he released an acoustic version of the song.[277] On September 27, Tesfaye released "Timeless", featuring previous collaborator Playboi Carti,[278] peaking at number three on the Billboard Hot 100 and becoming his highest debut on that chart to date.[279][280] On October 30, Tesfaye released "São Paulo", featuring Brazilian singer Anitta.[281] The album was released on January 31, 2025.[282] Upon release, the album debuted at number one on the Billboard 200 with 490,500 units, including 359,000 pure album sales, the latter figure being the highest for any male artist since 2020. With this feat, Hurry Up Tomorrow marks his fifth Billboard 200 chart-topping album and his highest opening week sales by overall units in the country. The album also marked the second largest week opener of 2025 (after Morgan Wallen's I'm the Problem), and the largest for any album since Taylor Swift's The Tortured Poets Department (2024).[283] On March 14, 2025, Tesfaye appeared on the US top five single "Rather Lie" from Playboi Carti's third studio album, Music.[284]

Tesfaye starred alongside Jenna Ortega in a companion film Hurry Up Tomorrow (2025) directed by Trey Edward Shults,[285][286] released on May 16[287] and widely criticized by critics, bombing at the box office bomb with grossing $7.6 million from its $15 million budget.[288][289] Brandon Yu of The New York Times described the psychological thriller as being "all style and no substance".[290] Charles Bramesco of IndieWire criticized Tesfaye's acting and writing, describing it as "modes of imitative, hollow performance, like a bad actor's varying notions of good acting".[291] Jordan Hoffman of Entertainment Weekly criticized the overall film as being a "nearly plot-free movie, [that] is self-indulgent, overly serious, and, worst of all, just plain dull."[292] IndieWire labeled it one of the "worst vanity projects ever made".[293] A more positive review came from G. Allen Johnson of the San Francisco Chronicle who described the film as "a risk-taking experience" and "visually marvelous as it is head-scratching", acknowledging it was imperfect, but applauding the way in which it "questions, probes and challenges viewers".[294] Having already set a new record for the top grossing tour by a male solo artist, on December 30, 2025, Tesfaye became the first artist on Spotify to have thirty songs with more than one billion streams each.[295][296]"""

tokens = text.encode("utf-8") # byte object
tokens = list(map(int, tokens)) # 0...255 range list of numbers

print(f"Length of text: {len(text)}")
print(f"Length of tokens: {len(tokens)}")

Length of text: 37137
Length of tokens: 37180


Length of tokens > Length of text because some special characters require more than 1 byte representation.

Eg -> an emoji may require 2 numbers to represent ([23, 45]) (Only an example, not real representation).

In [13]:
'''
This function returns the count of all pairs in the id list.
'''

def get_stats(ids):
    
    counts = {}
    
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
        
    return counts

stats = get_stats(tokens)
top_pair = max(stats, key=stats.get) # get the most frequent pair of tokens
top_pair

(101, 32)

In [14]:
'''
Merge function to remove the top pair and replace with a new token
'''

def merge(ids, pair, idx):
    
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(idx)
            i += 2
            # replacing the pair with new token
        else:
            new_ids.append(ids[i])
            i += 1
            # copying the original token
            
    return new_ids

'''
Now the only thing left is to repeat and re-iterate either until no frequent pair remains, or we define a stopping point.
We have to find a sweet spot between vocab size and sequence length that does not kill performance or make the computation expensive
'''

# Main loop

desired_size = 276 # desired_vocab_size
num_merges = desired_size - 256
ids = list(tokens)

merges = {}
for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    
    # because all nums from 0...255 are already covered by naive utf-8, any new token will start from 256
    print(f"Merging {pair} into new token {idx}")
    ids = merge(ids, pair, idx)
    merges[pair] = idx

Merging (101, 32) into new token 256
Merging (100, 32) into new token 257
Merging (116, 104) into new token 258
Merging (115, 32) into new token 259
Merging (110, 32) into new token 260
Merging (44, 32) into new token 261
Merging (101, 114) into new token 262
Merging (105, 110) into new token 263
Merging (116, 32) into new token 264
Merging (101, 257) into new token 265
Merging (258, 256) into new token 266
Merging (97, 114) into new token 267
Merging (97, 108) into new token 268
Merging (97, 110) into new token 269
Merging (101, 115) into new token 270
Merging (263, 103) into new token 271
Merging (111, 110) into new token 272
Merging (111, 114) into new token 273
Merging (121, 32) into new token 274
Merging (93, 32) into new token 275


Here we can see even some newly made tokens being merged into another new token, and that means now the vocabulary includes words.

Let's see the words.

In [15]:
vocab = {idx: bytes([idx]) for idx in range(256)}
# original vocab

print("Original utf-8 vocab: ")
print(vocab)

Original utf-8 vocab: 
{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'

In [16]:
for (p0, p1), idx in merges.items():
    # here (p0, p1) is the pair, idx is the replacement char
    vocab[idx] = vocab[p0] + vocab[p1]
    # byte concatenation (making words)
    
for key, value in vocab.items():
    print(f"Idx {key} | Byte {value} | Char {value.decode('utf-8', errors='replace')}")

Idx 0 | Byte b'\x00' | Char  
Idx 1 | Byte b'\x01' | Char 
Idx 2 | Byte b'\x02' | Char 
Idx 3 | Byte b'\x03' | Char 
Idx 4 | Byte b'\x04' | Char 
Idx 5 | Byte b'\x05' | Char 
Idx 6 | Byte b'\x06' | Char 
Idx 7 | Byte b'\x07' | Char 
Idx 8 | Byte b'\x08' | Char
Idx 9 | Byte b'\t' | Char 	
Idx 10 | Byte b'\n' | Char 

Idx 11 | Byte b'\x0b' | Char 
Idx 12 | Byte b'\x0c' | Char 
Idx 13 | Byte b'\r' | Char 
Idx 14 | Byte b'\x0e' | Char 
Idx 15 | Byte b'\x0f' | Char 
Idx 16 | Byte b'\x10' | Char 
Idx 17 | Byte b'\x11' | Char 
Idx 18 | Byte b'\x12' | Char 
Idx 19 | Byte b'\x13' | Char 
Idx 20 | Byte b'\x14' | Char 
Idx 21 | Byte b'\x15' | Char 
Idx 22 | Byte b'\x16' | Char 
Idx 23 | Byte b'\x17' | Char 
Idx 24 | Byte b'\x18' | Char 
Idx 25 | Byte b'\x19' | Char 
Idx 26 | Byte b'\x1a' | Char 
Idx 27 | Byte b'\x1b' | Char 
Idx 28 | Byte b'\x1c' | Char 
Idx 29 | Byte b'\x1d' | Char 
Idx 30 | Byte b'\x1e' | Char 
Idx 31 | Byte b'\x1f' | Char 
Idx 32 | Byte b' ' | Char  

Here in the end we see some words like 'the', 'in', 'an' being formed as a single token. Using huge data, many common words are replaced by a single token.


### NOTE:

Tokenization and tokenizer is a completely separate, independent module from the LLM. It can have its own training corpus (different from the language model). LLM only ever sees the tokens and never directly deals with any text.

![LLM and Tokenization](https://tse4.mm.bing.net/th/id/OIP.-SgizZ6_659ps321nxcHGwAAAA?r=0&pid=Api&P=0&h=180)

Lets now do the inference part, that is, decoding and encoding.

In [17]:
'''
We have already made the vocab and added the new tokens using the merges dictionary. 
'''

def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    # this is a byte stream (object)
    text = tokens.decode("utf-8", errors="replace")
    return text



Why errors replace? Because some particular sequence of bytes may not make a valid utf-8 representation.
Eg. 128 cannot be represented in utf-8, so simply decode gives an error, and 'replace' replaces it with Unicode replacement character (U+FFFD),
a placeholder.

In [18]:
'''
Encode function
'''

def encode(text):
    
    tokens = list(text.encode("utf-8"))
    
    while True:
        stats = get_stats(tokens)
        pair = min(stats, key = lambda p: merges.get(p, float("inf")))
        
        if pair not in merges:
            break
        
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
        
    return tokens

The line
> pair = min().......

Might seem a bit confusing, but the idea is that why are finding the pairs that should be merged first according to the merges dictionary. Why? because even some new tokens are replaced by another tokens, and for that reason, we need to go in order.
(For the detailed code explanation of the line, pls use chatgpt.)

Now let us test this, decode(encode(text)) == text should hold.

NOTE:
encode(decode([ids])) may not work for now, because of the errors=replace, but in the actual implementation, it should work.

In [19]:
example_text = "Hello, Hello, Hello!"

example_text == decode(encode(example_text))

True

In [20]:
'''
Lets try with a more complex sentence
'''

example_text_two = "hello 😀 this is a test"

tokens = encode(example_text_two)

print(decode(tokens) == example_text_two)
print(encode(decode(tokens)) == tokens)

True
True


We have now achieved an implementation of the simplest setting of tokenization. But there is a problem here.
In a very quantity of text, there are also many occurrences of the same words but before or after a new character, and this results in:

'dog', (dog), (dog!) and (dog?) all being represented by different tokens, which should not be the case.

(Read the GPT-2 paper for a deeper understanding.)

So, the fix is not allowing some pairs to merge, using a complicated regex.

Refer to [Openai's GPT-2 repo for the file for tokenization](https://github.com/openai/gpt-2/blob/master/src/encoder.py).

Here they use this regex (re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")) as a pre-processing step before sending the text to BPE for merging. Lets see what this does

In [21]:
%pip install regex
# this is a third-party regex library, not the inbuilt regex function
import regex as re 
pattern = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

text = "I've would've have, I could have, We're too deep in this, We'll be hurt, We'd never survive! 1234456, Hello1234565, ok "
re.findall(pattern, text)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


['I',
 "'ve",
 ' would',
 "'ve",
 ' have',
 ',',
 ' I',
 ' could',
 ' have',
 ',',
 ' We',
 "'re",
 ' too',
 ' deep',
 ' in',
 ' this',
 ',',
 ' We',
 "'ll",
 ' be',
 ' hurt',
 ',',
 ' We',
 "'d",
 ' never',
 ' survive',
 '!',
 ' 1234456',
 ',',
 ' Hello',
 '1234565',
 ',',
 ' ok',
 ' ']

's|'t|'re|'ve|'m|'ll|'d --> these patterns are used sequences like could've, we'd, he's, we'll, Model's to separate the actual word from the abbreviation or short form. Without this, in a huge quantity of text, would and would've can be 2 different tokens, even though their meaning is same.

?\p{L}+| means any letter of any language multiple times, but not whitespace.

?\p{N}+| means any number multiple times.

All of these help to separate words, whitespaces, numbers into chunks of text, and then each of these chunks is independently passed through the BPE algorithm, which prohibits some sequences of words into ever merging.

In [22]:
'''
Now lets look at tiktoken -> official gpt tokenizer, publically available only for inference, not training.
'''

# %pip install tiktoken

from tiktoken._educational import *

enc_gpt4 = SimpleBytePairEncoding.from_tiktoken("cl100k_base")
enc_gpt2 = SimpleBytePairEncoding.from_tiktoken("gpt2")
enc_gpt4.encode("hello world aaaaaaaaaaaa")
enc_gpt2.encode("hello world aaaaaaaaaaaa")

hello
hello
hello
hello
hello

 world
 world
 world
 world
 world
 world

 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa

hello
hello
hello
hello
hello

 world
 world
 world
 world
 world
 world

 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa
 aaaaaaaaaaaa



[31373, 995, 257, 24794, 24794, 46071]

Final part

in gpt-2, there are 50,257 tokens, 256 original, 50,000 merges, and one token added manually, which is '< |endoftext| >'.

This is added between documents, to signal the LLM that the document has ended, and the memory should be reset. 

GPT-4 adds 3 new tokens, which are fill in the middle [prefix, middle, and suffix].

This is very common in fine-tuning the base model to the 'instruct' model.